# Connect to the database

In [ ]:
import sqlalchemy
import pandas as pd
import numpy as np
from google.colab import userdata

In [ ]:
DB_USER = userdata.get('DB_USER')
DB_PASSWORD = userdata.get('DB_PASSWORD')
DB_HOST = userdata.get('DB_HOST')
DB_NAME = userdata.get('DB_NAME')
DB_PORT = "5432"

In [ ]:
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = sqlalchemy.create_engine(connection_string)

In [ ]:
pd.set_option('display.max_columns', None)

01/06

In [ ]:
# Something to present to Elie

abmf_hoffice_24_30_data = """
SELECT
    timestamp,
    gateway_serial,
    active_power_overall_total,
    apparent_power_overall_total,
    power_factor_overall,
    "total_system_kWh",
    load,
    frequency,
    active_assets,
    active_asset_count,
    workhour,
    transformer_capacity,
    transformer_load_percentage,
    line_to_neutral_voltage_phase_a,
    line_to_neutral_voltage_phase_b,
    line_to_neutral_voltage_phase_c,
    line_current_overall_phase_a,
    line_current_overall_phase_b,
    line_current_overall_phase_c,
    power_factor_overall_phase_a,
    power_factor_overall_phase_b,
    power_factor_overall_phase_c,
    voltage_unbalance_factor,
    current_unbalance_factor,
    total_harmonic_distortion_current_phase_a,
    total_harmonic_distortion_current_phase_b,
    total_harmonic_distortion_current_phase_c
FROM public.smart_device_readings
WHERE gateway_serial = 'EHM54090515' AND timestamp BETWEEN '2026-05-24 00:00:00' AND '2026-05-30 23:59:59'
ORDER BY timestamp DESC;
"""

abmf_hoffice_24_30_df = pd.read_sql(abmf_hoffice_24_30_data, engine)

In [ ]:
abmf_hoffice_24_30_df.head()

In [ ]:
abmf_hoffice_24_30_df['updated_power_factor_overall'] = abs(abmf_hoffice_24_30_df['active_power_overall_total'] / abmf_hoffice_24_30_df['apparent_power_overall_total'])
abmf_hoffice_24_30_df['updated_load_rate'] = abs(abmf_hoffice_24_30_df['apparent_power_overall_total'] * 100 / abmf_hoffice_24_30_df['updated_power_asset_capacity'])

In [ ]:
conditions = [
    abmf_hoffice_24_30_df['active_assets'] == 'Grid',
    abmf_hoffice_24_30_df['active_assets'] == 'Generator 1',
    abmf_hoffice_24_30_df['active_assets'] == ''
]

choices = [200, 88, 0]

abmf_hoffice_24_30_df['updated_power_asset_capacity'] = np.select(conditions, choices, default=0)

In [ ]:
abmf_hoffice_24_30_df.head()

In [ ]:
abmf_hoffice_24_30_df.updated_power_asset_capacity.value_counts()

In [ ]:
len(abmf_hoffice_24_30_df)

# Power Source Sessionalization

In [ ]:
abmf_hoffice_24_30_df_copy = abmf_hoffice_24_30_df.copy()

In [ ]:
# 1. Convert the column to datetime (just in case it loaded as a string/object)
abmf_hoffice_24_30_df_copy["timestamp"] = pd.to_datetime(abmf_hoffice_24_30_df_copy["timestamp"])

# 2. Set the column as the index
abmf_hoffice_24_30_df_copy = abmf_hoffice_24_30_df_copy.set_index("timestamp")

# 3. Crucial: Sort the index chronologically
abmf_hoffice_24_30_df_copy = abmf_hoffice_24_30_df_copy.sort_index()

In [ ]:
df_filled = abmf_hoffice_24_30_df_copy.resample("1min").first()

In [ ]:
df_filled.isna().sum()

In [ ]:
is_missing = df_filled["gateway_serial"].isna()
df_filled["session_marker"] = (is_missing != is_missing.shift()).cumsum()

outages_df = df_filled[is_missing].reset_index()
outages_df = outages_df.rename(columns={"index": "timestamp"})


# --- 4. Calculate Outage Durations ---
outage_analysis = (
    outages_df.groupby("session_marker")
    .agg(
        outage_start=("timestamp", "min"),
        outage_end=("timestamp", "max"),
        duration_minutes=("timestamp", "count"),
    )
    .reset_index(drop=True)
)


# --- 5. Asset Tracking: Before & After Lookups ---
# Look back 1 minute from start
pre_outage_timestamps = outage_analysis["outage_start"] - pd.Timedelta(minutes=1)
outage_analysis["asset_before_outage"] = pre_outage_timestamps.map(
    abmf_hoffice_24_30_df["active_assets"]
)

# Look forward 1 minute from end
post_outage_timestamps = outage_analysis["outage_end"] + pd.Timedelta(minutes=1)
outage_analysis["asset_after_outage"] = post_outage_timestamps.map(
    abmf_hoffice_24_30_df["active_assets"]
)


print("--- Final Power Outage Sessionization Report ---")
print(outage_analysis)

In [ ]:
outage_analysis[outage_analysis['duration_minutes'] != 1]

In [ ]:
abmf_hoffice_24_30_df.tail(30)

# Daily Load Plot and Weekly Overview

In [ ]:
# Useful Modules
import os
import zipfile
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import seaborn as sns

# turn 'timestamp' into a regular column
df = df_filled.reset_index()

# Rename index into timestamp if not currently nameed timestamp
if "timestamp" not in df.columns and "index" in df.columns:
    df = df.rename(columns={"index": "timestamp"})

# Clean, sort, and parse data
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Replace empty strings and NaNs with clear tracking labels
df["active_assets"] = df["active_assets"].replace(
    {"": "None / Standby", np.nan: "No Power"}
)

# Fill NaN load values with 0 to show a visible line during outages
df["load"] = df["load"].fillna(0)

# Minimum and max y_lim
raw_min = min(0, df["load"].min())
raw_max = df["load"].max()
y_span = raw_max - raw_min

# 5% margin
global_y_min = raw_min - (y_span * 0.05)
global_y_max = raw_max + (y_span * 0.05)

# A unique segment ID every time the asset switches
df["segment_id"] = (df["active_assets"] != df["active_assets"].shift()).cumsum()

# Explicit color mapping and strict order for perfect legend uniformity
hue_order = ["Grid", "Generator 1", "None / Standby", "No Power"]
custom_palette = {
    "Grid": "#1f77b4",  # Blue
    "Generator 1": "#ff7f0e",  # Orange
    "None / Standby": "#7f7f7f",  # Gray
    "No Power": "#000000",  # Black
}

# List to track filenames for zipping
generated_files = []

# Loop through each available day to create individual plots
for current_date, day_df in df.groupby(df["timestamp"].dt.date):
    day_df = day_df.copy()

    # Standardize all times onto a single hidden dummy day for smooth 24-hour scaling
    day_df["time_axis"] = pd.to_datetime("2000-01-01 " + day_df["timestamp"].dt.strftime("%H:%M:%S"))

    # Dynamically find the day of the week (e.g., "Wednesday")
    day_of_week = pd.to_datetime(current_date).strftime("%A")

    # Initialize figure using subplots
    fig, ax = plt.subplots(figsize=(15, 6))

    # Plot daily load lines using the normalized time_axis
    sns.lineplot(
        data=day_df,
        x="time_axis",
        y="load",
        hue="active_assets",
        hue_order=hue_order,

        units="segment_id",
        estimator=None,
        palette=custom_palette,
        linewidth=2,
        alpha=0.8,
        ax=ax,
    )

    # Force x-axis labels to strictly display HH:MM from 00:00 to 23:59
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))  # Tick every 2 hours
    ax.set_xlim(pd.to_datetime('2000-01-01 00:00:00'), pd.to_datetime('2000-01-01 23:59:00'))

    # Apply the uniform, padded y-limits to this plot
    ax.set_ylim(global_y_min, global_y_max)

    # Formatting and Labels
    ax.set_title(
        f"ABMF Load Profile — {day_of_week}, {current_date}",
        fontsize=14,
        pad=15,
        weight="bold",
    )
    ax.set_xlabel("Hour of Day", fontsize=12)
    ax.set_ylabel("Load (kW)", fontsize=12)
    ax.grid(True, linestyle=":", alpha=0.5)

    # Deduplicate legend entries cleanly while honoring hue_order positioning
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(
        [by_label[lbl] for lbl in hue_order if lbl in by_label],
        [lbl for lbl in hue_order if lbl in by_label],
        title="System Labels",
        loc="upper right",
    )

    plt.tight_layout()

    # Save filename temporarily
    filename = f"load_profile_{current_date}.png"
    plt.savefig(filename, dpi=300)
    generated_files.append(filename)

    # Free background figure memory
    plt.close(fig)

# Generate the Full Week Profile Plot
start_date = df["timestamp"].min().strftime("%Y-%m-%d")
end_date = df["timestamp"].max().strftime("%Y-%m-%d")

fig, ax = plt.subplots(figsize=(15, 6))

# Plot full weekly dataset sequentially (using real timestamps)
sns.lineplot(
    data=df,
    x="timestamp",
    y="load",
    hue="active_assets",
    hue_order=hue_order,
    units="segment_id",
    estimator=None,
    palette=custom_palette,
    linewidth=1.5,  # Thinner lines look better with high density full-week data
    alpha=0.8,
    ax=ax,
)

# Format continuous timeline for x-axis ticks at each day boundary
ax.xaxis.set_major_formatter(mdates.DateFormatter('%a, %b %d'))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
ax.set_xlim(df["timestamp"].min(), df["timestamp"].max())

# Apply the exact same padded uniform y-limits for seamless benchmarking
ax.set_ylim(global_y_min, global_y_max)

# Sessionalization Title and Labels
ax.set_title(
    f"ABMF Load Profile — {start_date} till {end_date}",
    fontsize=14,
    pad=15,
    weight="bold",
)
ax.set_xlabel("Day of Week", fontsize=12)
ax.set_ylabel("Load (kW)", fontsize=12)
ax.grid(True, linestyle=":", alpha=0.5)

# Deduplicate and arrange full-week legend identically
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(
    [by_label[lbl] for lbl in hue_order if lbl in by_label],
    [lbl for lbl in hue_order if lbl in by_label],
    title="System Labels",
    loc="upper right",
)

plt.tight_layout()

# Save the full week chart
full_week_filename = f"ABMF Load Profile - {start_date} till {end_date}.png"
plt.savefig(full_week_filename, dpi=300)
generated_files.append(full_week_filename)

plt.close(fig)

# Pack all daily plots + full week plot into a single zip file
zip_filename = "load_profiles.zip"
with zipfile.ZipFile(zip_filename, "w") as zipf:
    for file in generated_files:
        zipf.write(file)
        os.remove(file)  # Clean up loose PNG files after compilation

print(
    f"All done. Saved into '{zip_filename}'."
)

# Load Rate

In [ ]:
df_filled['updated_load_rate'] = df_filled['apparent_power_overall_total'] * 100 / df_filled['updated_power_asset_capacity']

In [ ]:
# Useful Modules
import os
import zipfile
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import seaborn as sns

# turn 'timestamp' into a regular column
df = df_filled.reset_index()

# Rename index into timestamp if not currently named timestamp
if "timestamp" not in df.columns and "index" in df.columns:
    df = df.rename(columns={"index": "timestamp"})

# Clean, sort, and parse data
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Replace empty strings and NaNs with clear tracking labels
df["active_assets"] = df["active_assets"].replace(
    {"": "None / Standby", np.nan: "No Power"}
)

# --- THE FIX: Convert infinities to NaN, then fill both with 0 ---
df["updated_load_rate"] = df["updated_load_rate"].replace([np.inf, -np.inf], np.nan)
df["updated_load_rate"] = df["updated_load_rate"].fillna(0)
# -----------------------------------------------------------------

# Minimum and max y_lim
raw_min = min(0, df["updated_load_rate"].min())
raw_max = df["updated_load_rate"].max()
y_span = raw_max - raw_min

# 5% margin
global_y_min = raw_min - (y_span * 0.05)
global_y_max = raw_max + (y_span * 0.05)

# A unique segment ID every time the asset switches
df["segment_id"] = (df["active_assets"] != df["active_assets"].shift()).cumsum()

# Explicit color mapping and strict order for perfect legend uniformity
hue_order = ["Grid", "Generator 1", "None / Standby", "No Power"]
custom_palette = {
    "Grid": "#1f77b4",  # Blue
    "Generator 1": "#ff7f0e",  # Orange
    "None / Standby": "#7f7f7f",  # Gray
    "No Power": "#000000",  # Black
}

# List to track filenames for zipping
generated_files = []

# Loop through each available day to create individual plots
for current_date, day_df in df.groupby(df["timestamp"].dt.date):
    day_df = day_df.copy()

    # Standardize all times onto a single hidden dummy day for smooth 24-hour scaling
    day_df["time_axis"] = pd.to_datetime("2000-01-01 " + day_df["timestamp"].dt.strftime("%H:%M:%S"))

    # Dynamically find the day of the week (e.g., "Wednesday")
    day_of_week = pd.to_datetime(current_date).strftime("%A")

    # Initialize figure using subplots
    fig, ax = plt.subplots(figsize=(15, 6))

    # Plot daily load rate lines using the normalized time_axis
    sns.lineplot(
        data=day_df,
        x="time_axis",
        y="updated_load_rate",
        hue="active_assets",
        hue_order=hue_order,
        units="segment_id",
        estimator=None,
        palette=custom_palette,
        linewidth=2,
        alpha=0.8,
        ax=ax,
    )

    # Force x-axis labels to strictly display HH:MM from 00:00 to 23:59
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))  # Tick every 2 hours
    ax.set_xlim(pd.to_datetime('2000-01-01 00:00:00'), pd.to_datetime('2000-01-01 23:59:00'))

    # Apply the uniform, padded y-limits to this plot
    ax.set_ylim(global_y_min, global_y_max)

    # Formatting and Labels
    ax.set_title(
        f"ABMF Load Rate Profile — {day_of_week}, {current_date}",
        fontsize=14,
        pad=15,
        weight="bold",
    )
    ax.set_xlabel("Hour of Day", fontsize=12)
    ax.set_ylabel("Load Rate (%)", fontsize=12)
    ax.grid(True, linestyle=":", alpha=0.5)

    # Deduplicate legend entries cleanly while honoring hue_order positioning
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(
        [by_label[lbl] for lbl in hue_order if lbl in by_label],
        [lbl for lbl in hue_order if lbl in by_label],
        title="System Labels",
        loc="upper right",
    )

    plt.tight_layout()

    # Save filename temporarily
    filename = f"load_rate_profile_{current_date}.png"
    plt.savefig(filename, dpi=300)
    generated_files.append(filename)

    # Free background figure memory
    plt.close(fig)

# Generate the Full Week Profile Plot
start_date = df["timestamp"].min().strftime("%Y-%m-%d")
end_date = df["timestamp"].max().strftime("%Y-%m-%d")

fig, ax = plt.subplots(figsize=(15, 6))

# Plot full weekly dataset sequentially (using real timestamps)
sns.lineplot(
    data=df,
    x="timestamp",
    y="updated_load_rate",
    hue="active_assets",
    hue_order=hue_order,
    units="segment_id",
    estimator=None,
    palette=custom_palette,
    linewidth=1.5,  # Thinner lines look better with high density full-week data
    alpha=0.8,
    ax=ax,
)

# Format continuous timeline for x-axis ticks at each day boundary
ax.xaxis.set_major_formatter(mdates.DateFormatter('%a, %b %d'))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
ax.set_xlim(df["timestamp"].min(), df["timestamp"].max())

# Apply the exact same padded uniform y-limits for seamless benchmarking
ax.set_ylim(global_y_min, global_y_max)

# Sessionalization Title and Labels
ax.set_title(
    f"ABMF Load Rate Profile — {start_date} till {end_date}",
    fontsize=14,
    pad=15,
    weight="bold",
)
ax.set_xlabel("Day of Week", fontsize=12)
df.rename(columns={"load": "updated_load_rate"}, inplace=True)
ax.set_ylabel("Load Rate (%)", fontsize=12)
ax.grid(True, linestyle=":", alpha=0.5)

# Deduplicate and arrange full-week legend identically
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(
    [by_label[lbl] for lbl in hue_order if lbl in by_label],
    [lbl for lbl in hue_order if lbl in by_label],
    title="System Labels",
    loc="upper right",
)

plt.tight_layout()

# Save the full week chart
full_week_filename = f"ABMF Load Rate Profile - {start_date} till {end_date}.png"
plt.savefig(full_week_filename, dpi=300)
generated_files.append(full_week_filename)

plt.close(fig)

# Pack all daily plots + full week plot into a single zip file
zip_filename = "load_rate_profiles.zip"
with zipfile.ZipFile(zip_filename, "w") as zipf:
    for file in generated_files:
        zipf.write(file)
        os.remove(file)  # Clean up loose PNG files after compilation

print(
    f"All done. Saved into '{zip_filename}'."
)

Seems the load rate is not a accurate measure of consumption

In [ ]:
abmf_hoffice_24_30_df_copy['active_power_overall_total'].nsmallest(2)

In [ ]:
abmf_hoffice_24_30_df[abmf_hoffice_24_30_df['active_power_overall_total'] < 0]

In [ ]:
len(abmf_hoffice_24_30_df[abmf_hoffice_24_30_df['active_power_overall_total'] < 0])

In [ ]:
abmf_hoffice_24_30_df_copy['active_power_overall_total'].nsmallest(2).iloc[-1]

In [ ]:
abmf_hoffice_24_30_df_copy['active_power_overall_total'].nsmallest(23).iloc[-10]

In [ ]:
abmf_hoffice_24_30_df['frequency'].hist()

In [ ]:
abmf_hoffice_24_30_df.active_assets.value_counts()

In [ ]:
# For generators
abmf_hoffice_24_30_df[abmf_hoffice_24_30_df['active_assets'] == 'Generator 1']['frequency'].hist()

In [ ]:
# For grid
abmf_hoffice_24_30_df[abmf_hoffice_24_30_df['active_assets'] == 'Grid']['frequency'].hist()

In [ ]:
# Filter for finite values only, then plot
pf_data = abmf_hoffice_24_30_df["updated_power_factor_overall"]
pf_data[np.isfinite(pf_data)].hist(bins=50)

In [ ]:
generator_1_df = abmf_hoffice_24_30_df[
    abmf_hoffice_24_30_df["active_assets"] == "Generator 1"
]

pf_data = generator_1_df["updated_power_factor_overall"]

pf_data[np.isfinite(pf_data)].hist(bins=50)

In [ ]:
grid_df = abmf_hoffice_24_30_df[
    abmf_hoffice_24_30_df["active_assets"] == "Grid"
]

pf_data = grid_df["updated_power_factor_overall"]

pf_data[np.isfinite(pf_data)].hist(bins=50)

In [ ]:
abmf_hoffice_24_30_df[abmf_hoffice_24_30_df['updated_power_factor_overall'] > 1]

In [ ]:
abmf_hoffice_24_30_df["updated_power_factor_overall"]

In [ ]:
abmf_hoffice_24_30_df['updated_power_factor_overall'].hist()

In [ ]:
abmf_hoffice_24_30_df['updated_power_factor_overall'].max()

In [ ]:
abmf_hoffice_24_30_df[abmf_hoffice_24_30_df['updated_power_factor_overall'] < 0]

In [ ]:
(abmf_hoffice_24_30_df[['line_current_overall_phase_a',
    'line_current_overall_phase_b',
    'line_current_overall_phase_c']]).min

In [ ]:
phase_cols = [
    "line_current_overall_phase_a",
    "line_current_overall_phase_b",
    "line_current_overall_phase_c",
]

In [ ]:
absolute_min = (
    abmf_hoffice_24_30_df[phase_cols]
    .where(abmf_hoffice_24_30_df[phase_cols] != 0)
    .min()
    .min()
)
print(absolute_min)

In [ ]:
phase_cols = [
    "line_current_overall_phase_a",
    "line_current_overall_phase_b",
    "line_current_overall_phase_c",
]

# .where() keeps values if the condition is True, otherwise makes them NaN
min_per_column = (
    abmf_hoffice_24_30_df[phase_cols]
    .where(abmf_hoffice_24_30_df[phase_cols] != 0)
    .min()
)
print(min_per_column)

In [ ]:
# Current data

all_current_data = """
SELECT
    line_current_overall_phase_a,
    line_current_overall_phase_b,
    line_current_overall_phase_c
FROM public.smart_device_readings
WHERE gateway_serial = 'EHM54090515'
ORDER BY timestamp DESC;
"""

all_current_df = pd.read_sql(all_current_data, engine)

In [ ]:
min_per_column = (
    all_current_df
    .where(all_current_df[phase_cols] != 0)
    .min()
)
print(min_per_column)

In [ ]:
# Current data

all_current_data_aba = """
SELECT
    line_current_overall_phase_a,
    line_current_overall_phase_b,
    line_current_overall_phase_c
FROM public.smart_device_readings
WHERE gateway_serial = 'EHM21120502'
ORDER BY timestamp DESC;
"""

all_current_df_aba = pd.read_sql(all_current_data_aba, engine)

In [ ]:
min_per_column = (
    all_current_df_aba
    .where(all_current_df_aba[phase_cols] != 0)
    .min()
)
print(min_per_column)

In [ ]:
# ABA

In [ ]:
# Something to present to Elie

abmf_hoffice_31_01_data = """
SELECT
    timestamp,
    gateway_serial,
    active_power_overall_total,
    apparent_power_overall_total,
    power_factor_overall,
    "total_system_kWh",
    load,
    frequency,
    active_assets,
    active_asset_count,
    workhour,
    transformer_capacity,
    transformer_load_percentage,
    line_to_neutral_voltage_phase_a,
    line_to_neutral_voltage_phase_b,
    line_to_neutral_voltage_phase_c,
    line_current_overall_phase_a,
    line_current_overall_phase_b,
    line_current_overall_phase_c,
    power_factor_overall_phase_a,
    power_factor_overall_phase_b,
    power_factor_overall_phase_c,
    voltage_unbalance_factor,
    current_unbalance_factor,
    total_harmonic_distortion_current_phase_a,
    total_harmonic_distortion_current_phase_b,
    total_harmonic_distortion_current_phase_c
FROM public.smart_device_readings
WHERE gateway_serial = 'EHM54090515' AND timestamp BETWEEN '2026-05-31 00:00:00' AND '2026-06-01 14:54:00'
ORDER BY timestamp DESC;
"""

abmf_hoffice_31_01_df = pd.read_sql(abmf_hoffice_31_01_data, engine)

In [ ]:
abmf_hoffice_31_01_df.head()

In [ ]:
# Something to present to Elie

abmf_aba_all = """
SELECT
    active_assets,
    active_asset_count,
    line_to_neutral_voltage_phase_a,
    line_to_neutral_voltage_phase_b,
    line_to_neutral_voltage_phase_c,
    line_current_overall_phase_a,
    line_current_overall_phase_b,
    line_current_overall_phase_c,
    input_channel_1_current,
    input_channel_2_current,
    input_channel_3_current,
    input_channel_4_current,
    input_channel_5_current,
    input_channel_6_current
FROM public.smart_device_readings
WHERE gateway_serial = 'EHM21120502'
ORDER BY timestamp DESC;
"""

abmf_aba_all_df = pd.read_sql(abmf_aba_all, engine)

In [ ]:
abmf_aba_all_df['active_assets'].value_counts()

In [ ]:
both_asset_active = abmf_aba_all_df[abmf_aba_all_df['active_assets'] == 'Grid,Generator 1']

In [ ]:
# Now, max
min_per_column = (
    both_asset_active[['input_channel_1_current',
    'input_channel_2_current',
    'input_channel_3_current',
    'input_channel_4_current',
    'input_channel_5_current',
    'input_channel_6_current']]
    .max()
)
print(min_per_column)

In [ ]:
channels = [
    "input_channel_1_current",
    "input_channel_2_current",
    "input_channel_3_current",
    "input_channel_4_current",
    "input_channel_5_current",
    "input_channel_6_current",
]

# .where() turns 0s into NaNs independently, which .min() automatically ignores
min_per_column = (
    both_asset_active[channels].where(both_asset_active[channels] != 0).min()
)

print(min_per_column)

In [ ]:
min_per_column = (
    abmf_aba_all_df[['input_channel_1_current',
    'input_channel_2_current',
    'input_channel_3_current',
    'input_channel_4_current',
    'input_channel_5_current',
    'input_channel_6_current']]
    .max()
)
print(min_per_column)

In [ ]:
channels = [
    "input_channel_1_current",
    "input_channel_2_current",
    "input_channel_3_current",
    "input_channel_4_current",
    "input_channel_5_current",
    "input_channel_6_current",
]

# .where() turns 0s into NaNs independently, which .min() automatically ignores
min_per_column = (
    abmf_aba_all_df[channels].where(abmf_aba_all_df[channels] != 0).min()
)

print(min_per_column)

In [ ]:
# Filter for rows where channel 5 is not zero, then find the minimum
min_channel_5 = both_asset_active[both_asset_active['input_channel_4_current'] != 0]['input_channel_4_current'].min()

print(min_channel_5)

In [ ]:
min_channel_4 = (
    both_asset_active["input_channel_4_current"]
    .where(both_asset_active["input_channel_4_current"] != 0)
    .min()
)

# Convert a NaN result to 0 if the column had no valid non-zero data
import pandas as pd

if pd.isna(min_channel_4):
    min_channel_4 = 0.0

print(min_channel_4)

In [ ]:
both_asset_active['input_channel_4_current'].hist()

In [ ]:
both_asset_active['input_channel_5_current'].hist()

In [ ]:
both_asset_active['input_channel_6_current'].hist()

02/06

In [ ]:
# Something to present to Elie

abmf_hoffice_24_30_data = """
SELECT
    timestamp,
    gateway_serial,
    active_power_overall_total,
    apparent_power_overall_total,
    power_factor_overall,
    "total_system_kWh",
    load,
    frequency,
    active_assets,
    active_asset_count,
    workhour,
    transformer_capacity,
    transformer_load_percentage,
    line_to_neutral_voltage_phase_a,
    line_to_neutral_voltage_phase_b,
    line_to_neutral_voltage_phase_c,
    line_current_overall_phase_a,
    line_current_overall_phase_b,
    line_current_overall_phase_c,
    power_factor_overall_phase_a,
    power_factor_overall_phase_b,
    power_factor_overall_phase_c,
    voltage_unbalance_factor,
    current_unbalance_factor,
    total_harmonic_distortion_current_phase_a,
    total_harmonic_distortion_current_phase_b,
    total_harmonic_distortion_current_phase_c
FROM public.smart_device_readings
WHERE gateway_serial = 'EHM54090515' AND timestamp BETWEEN '2026-05-24 00:00:00' AND '2026-05-30 23:59:59'
ORDER BY timestamp DESC;
"""

abmf_hoffice_24_30_df = pd.read_sql(abmf_hoffice_24_30_data, engine)